In [0]:
from pyspark.sql.functions import to_date

# Step 1：建立原始資料（與前面保持一致）
data = [
    ("001", "Alice", "A", 100, "2024-01-01"),
    ("002", "Bob", "A", 200, "2024-01-02"),
    ("003", "Cathy", "B", 150, "2024-01-03"),
    ("004", "David", "B", 300, "2024-01-04"),
    ("005", "Eve", "C", 250, "2024-01-05"),
    ("006", "Frank", "C", 180, "2024-01-06"),
    ("007", "Grace", "A", 220, "2024-01-07"),
    ("008", "Henry", "B", 90, "2024-01-08"),
    ("009", "Ivy", "C", 130, "2024-01-09"),
    ("010", "Jack", "A", 170, "2024-01-10"),
]

df = spark.createDataFrame(
    data,
    ["id", "name", "category", "amount", "date"]
)

# 轉成時間型態
df = df.withColumn("date", to_date("date"))

df_time = df.groupBy("date").sum("amount") \
    .withColumnRenamed("sum(amount)", "total_amount")


display(df_time)

# Step 2：資料清理（Cleaning）
df_clean = df.fillna({
    "name": "UNKNOWN",
    "category": "UNKNOWN",
    "amount": 0
})

# Step 3：資料轉換（Transformation）
from pyspark.sql.functions import col

df_result = df_clean.withColumn(
    "amount_with_tax",
    col("amount") * 1.05
)

# Step 4：資料分析（Aggregation）
df_agg = df_result.groupBy("category").sum("amount") \
    .withColumnRenamed("sum(amount)", "total_amount")

# Step 5：顯示結果
print("=== 清理後資料 ===")
display(df_clean)

print("=== 加上稅後金額 ===")
display(df_result)

print("=== 類別統計 ===")
display(df_agg)

# Step 6：將結果寫入 Databricks table，供 Dashboard / SQL 使用

df_result.write.mode("overwrite").saveAsTable("order_result")
df_agg.write.mode("overwrite").saveAsTable("order_category_summary")
df_time.write.mode("overwrite").saveAsTable("order_time_summary")

print("=== 已成功寫入 tables ===")
print("1. order_result")
print("2. order_category_summary")
print("3. order_time_summary")

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.